In [ ]:
from presidio_evaluator import InputSample

train_samples = InputSample.read_dataset_json("../data/train_May-04-2026.json")
test_samples = InputSample.read_dataset_json("../data/test_May-04-2026.json")

print(f"train: {len(train_samples)} 筆")
print(f"test: {len(test_samples)} 筆")

# 檢查前幾筆有沒有正常解析
for s in train_samples[:2]:
    print(s.full_text)
    print(s.tags)
    print("---")

In [ ]:
from spacy.tokens import DocBin
db = DocBin().from_disk("train_en.spacy")
docs = list(db.get_docs(nlp.vocab))

# 看前3筆
for doc in docs[:3]:
    print(f"文字: {doc.text}")
    print(f"ents: {list(doc.ents)}")
    print(f"tokens: {[t.text for t in doc]}")
    print("---")
for s in train_samples[:3]:
    print(f"full_text: {s.full_text}")
    print(f"tokens: {[t.text for t in s.tokens] if s.tokens else 'None'}")
    print(f"tags: {s.tags}")
    print("---")

In [ ]:
import spacy
from spacy.tokens import DocBin, Doc

nlp = spacy.blank("en")
def samples_to_docbin_from_tags(samples, output_path):
    db = DocBin()
    skipped = 0

    for sample in samples:
        if not sample.tokens or not sample.tags:
            skipped += 1
            continue
        
        words = [t.text for t in sample.tokens]
        spaces = [t.whitespace_ != "" for t in sample.tokens]
        doc = Doc(nlp.vocab, words=words, spaces=spaces)
        
        ents = []
        start = None
        current_label = None
        
        for i, tag in enumerate(sample.tags):
            # 判斷是 BIO 格式還是純標籤格式
            if tag.startswith("B-"):
                # BIO 格式：B- 開頭
                if start is not None:
                    ents.append(spacy.tokens.Span(doc, start, i, label=current_label))
                start = i
                current_label = tag[2:]
            elif tag.startswith("I-"):
                pass  # 繼續同一實體
            elif tag == "O":
                # 實體結束
                if start is not None:
                    ents.append(spacy.tokens.Span(doc, start, i, label=current_label))
                    start = None
                    current_label = None
            else:
                # 純標籤格式（如 "PERSON", "GPE"）
                if tag == current_label:
                    pass  # 同一實體繼續
                else:
                    # 前一個實體結束，新實體開始
                    if start is not None:
                        ents.append(spacy.tokens.Span(doc, start, i, label=current_label))
                    start = i
                    current_label = tag
        
        # 處理最後一個實體
        if start is not None:
            ents.append(spacy.tokens.Span(doc, start, len(doc), label=current_label))
        
        doc.ents = spacy.util.filter_spans(ents)
        db.add(doc)
    
    db.to_disk(output_path)
    print(f"✅ 儲存 {len(samples) - skipped} 筆 → {output_path}（跳過 {skipped} 筆）")

# 重新轉換
samples_to_docbin_from_tags(train_samples, "train_en.spacy")
samples_to_docbin_from_tags(test_samples, "dev_en.spacy")



In [ ]:
# InputSample.create_spacy_dataset(train_samples, output_path="train_en.spacy")
# InputSample.create_spacy_dataset(test_samples, output_path="dev_en.spacy")

In [ ]:
from spacy.tokens import DocBin
import spacy
from collections import Counter

nlp = spacy.blank("en")

# 讀回來檢查
for path, name in [("train_en.spacy", "Train"), ("dev_en.spacy", "Dev")]:
    db = DocBin().from_disk(path)
    docs = list(db.get_docs(nlp.vocab))
    
    label_counter = Counter()
    for doc in docs:
        for ent in doc.ents:
            label_counter[ent.label_] += 1
    
    print(f"=== {name} ===")
    print(f"總筆數: {len(docs)}")
    print(f"實體分布: {dict(label_counter)}")
    print()

In [ ]:
cd github-star/presidio-research\app
python -m spacy init config config_en.cfg --lang en --pipeline ner --optimize accuracy --force
python -m spacy train config_en.cfg --output ./output --paths.train ./train_en.spacy --paths.dev ./dev_en.spacy
@REM python -m spacy train config_en.cfg --output ./output_en --paths.train ./train.spacy --paths.dev ./dev.spacy --training.max_steps 200 --training.eval_frequency 20 --training.patience 0

In [ ]:
cd presidio-research\app
python -m spacy init config config.cfg --lang en --pipeline ner
python -m spacy train config.cfg  --output ./output   --paths.train train_en.spacy  --paths.dev dev_en.spacy

In [ ]:
E    #       LOSS TOK2VEC  LOSS NER  ENTS_F  ENTS_P  ENTS_R  SCORE 
---  ------  ------------  --------  ------  ------  ------  ------
  0       0          0.00     61.09    0.00    0.00    0.00    0.00
  1     200        106.66   3096.88   54.64   56.59   52.82    0.55
  2     400        346.60   2318.25   64.98   65.24   64.72    0.65
  3     600        181.36   1460.87   57.74   57.23   58.27    0.58
  5     800        305.39   1425.22   65.51   66.32   64.72    0.66
  8    1000        152.91    623.22   65.73   65.53   65.93    0.66
 11    1200         91.14    213.31   66.67   69.43   64.11    0.67
 15    1400         67.10    111.47   66.41   64.22   68.75    0.66
 19    1600         77.28    123.54   66.67   66.01   67.34    0.67
 25    1800        217.15    286.65   68.65   70.02   67.34    0.69
 32    2000         73.37     67.27   64.67   62.04   67.54    0.65
 40    2200        136.86     78.81   69.65   71.89   67.54    0.70
 50    2400        140.23    110.98   68.41   71.09   65.93    0.68
 61    2600        673.97    563.81   70.09   67.41   72.98    0.70
 71    2800        153.34    114.83   68.80   68.25   69.35    0.69
 82    3000         44.31     28.59   72.02   69.96   74.19    0.72
 92    3200        391.48    179.15   68.91   67.70   70.16    0.69
103    3400       4918.49   1736.50   70.86   71.37   70.36    0.71
113    3600         40.16     29.86   70.39   72.34   68.55    0.70
124    3800          3.10      2.21   71.24   70.33   72.18    0.71
134    4000          7.02      3.43   72.47   73.60   71.37    0.72
145    4200          0.00      0.00   71.02   71.90   70.16    0.71
155    4400          0.00      0.00   70.55   71.58   69.56    0.71
166    4600          0.83      1.32   72.69   71.62   73.79    0.73
176    4800         34.13     11.96   73.84   72.34   75.40    0.74
187    5000       1106.49    568.26   61.95   66.13   58.27    0.62
197    5200        327.05    190.90   69.42   73.05   66.13    0.69
208    5400        156.59     73.96   65.53   69.37   62.10    0.66
219    5600         63.03     23.32   67.41   67.47   67.34    0.67
229    5800         62.47     21.32   72.24   71.32   73.19    0.72
240    6000         60.86     19.84   73.47   75.91   71.17    0.73
250    6200        315.96    107.62   74.40   73.81   75.00    0.74
261    6400         27.92      8.49   74.53   77.27   71.98    0.75
271    6600        205.07     63.99   70.37   73.20   67.74    0.70
282    6800        521.33    105.03   72.93   73.08   72.78    0.73
292    7000        369.74     89.02   71.31   71.46   71.17    0.71
303    7200        275.01     57.69   67.75   67.95   67.54    0.68
313    7400       4440.60    801.83   68.56   68.98   68.15    0.69
324    7600        237.72     63.80   65.80   71.03   61.29    0.66
334    7800         45.38     11.79   67.43   69.91   65.12    0.67
345    8000          0.01      0.00   67.71   70.28   65.32    0.68

In [ ]:
import spacy

nlp = spacy.load("./output/model-best")

print(nlp.get_pipe("ner").labels)

In [ ]:
from spacy.tokens import DocBin
db = DocBin().from_disk("train_en.spacy")
docs = list(db.get_docs(nlp.vocab))

print(f"共 {len(docs)} 筆")
for doc in docs[:3]:
    print(doc.text)
    print([(ent.text, ent.label_) for ent in doc.ents])
    print("---")

In [ ]:
import spacy

nlp = spacy.load("./output/model-best")

text = "Justin sent his email test@example.com to Kat in 6/8/2023 and she replied to him on 6/9/2023. They also discussed the new project at 10:00 AM."
doc = nlp(text)

for ent in doc.ents:
    print(ent.text, "->", ent.label_)

In [ ]:
import time
from datetime import datetime

import pyperclip
import spacy

# -----------------------------
# Load custom spaCy model
# -----------------------------
MODEL_PATH = "./output/model-best"

print("=" * 60)
print("Loading custom model...")
nlp = spacy.load(MODEL_PATH)
print("Model loaded successfully.")
print("NER Labels:", nlp.get_pipe("ner").labels)
print("=" * 60)

last_text = ""


def anonymize(text, entities):
    """
    Replace entities with their labels.
    """
    result = text

    # Replace from back to front to avoid index shifting
    for ent in sorted(entities, key=lambda x: x.start_char, reverse=True):
        replacement = f"<{ent.label_}>"
        result = (
            result[:ent.start_char]
            + replacement
            + result[ent.end_char:]
        )

    return result


print("Clipboard monitor started...")
print("Press Ctrl+C to stop.\n")

while True:

    try:
        current = pyperclip.paste()

        if current != last_text and current.strip():

            last_text = current

            print("\n" + "=" * 70)
            print("Clipboard Updated")
            print("=" * 70)
            print("Time :", datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
            print()

            print("Original Text")
            print("-" * 70)
            print(current)
            print()

            start = time.perf_counter()

            doc = nlp(current)

            elapsed = (time.perf_counter() - start) * 1000

            print("Detected Entities")
            print("-" * 70)

            if len(doc.ents) == 0:
                print("No entities detected.")

            else:

                statistics = {}

                for i, ent in enumerate(doc.ents, start=1):

                    statistics[ent.label_] = statistics.get(ent.label_, 0) + 1

                    print(f"Entity #{i}")
                    print(f"Text       : {ent.text}")
                    print(f"Label      : {ent.label_}")
                    print(f"Start      : {ent.start_char}")
                    print(f"End        : {ent.end_char}")
                    print()

                print("-" * 70)
                print("Statistics")

                for label, count in statistics.items():
                    print(f"{label:10} : {count}")

            print()

            anonymized = anonymize(current, doc.ents)

            print("-" * 70)
            print("Anonymized Text")
            print("-" * 70)
            print(anonymized)
            print()

            pyperclip.copy(anonymized)

            print("Clipboard updated successfully.")
            print(f"NER Processing Time : {elapsed:.2f} ms")
            print("=" * 70)

        time.sleep(0.3)

    except KeyboardInterrupt:
        print("\nProgram stopped.")
        break

    except Exception as e:
        print(e)
        time.sleep(1)